# Studio 2 — Pelatihan di Colab (runner)

Notebook ini **tipis**: semua logika ada di repo (`train.py`, `make_split.py`, `evaluate.py`), notebook hanya menyiapkan GPU + dataset lalu menjalankan skrip yang sama dengan yang dipakai asisten.

Batas dari halaman studio: `model.pth` **≤ 500 MB**, evaluasi seluruh validation set **≤ 15 menit**; bobot > 100 MB harus lewat GitHub Release / LFS. Yang dinilai asisten = **commit terakhir di branch utama sebelum tenggat**.

Alur: mount Drive -> clone repo -> temukan `train/` -> **cek split beku** -> latih -> uji `evaluate.py` -> unduh `submission/`.


In [ ]:
REPO_URL = "https://github.com/stcahya/DL02-klasifikasi-sampah.git"
BRANCH   = "main"

# Folder Drive hasil ekstrak dataset dari folder kelas (harus berisi train/).
DRIVE_ROOT = "/content/drive/MyDrive/Studio2-DeepLearning"

# Yang dilatih = KANDIDAT di experiments/, bukan submission/.
# submission/ tetap berisi pasangan ResNet-18 yang sudah bisa dievaluasi (0,9373 di my_val).
CANDIDATE   = "experiments/ensemble_model.py"
OUT_WEIGHTS = "runs/ens_20ep.pth"

EPOCHS, BATCH_SIZE, WORKERS = 20, 64, 4


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content
![ -d repo/.git ] || git clone -b {BRANCH} {REPO_URL} repo
!cd repo && git pull --ff-only
%cd /content/repo
!pip install -q -r submission/requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "TANPA GPU - set Runtime > Change runtime type > T4 GPU"
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"


In [ ]:
import pathlib

CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]
# Jumlah gambar snapshot dataset yang dipakai membekukan splits/val.txt (make_split --check harus [OK]).
EXPECT = {"botol_plastik": 523, "gelas_plastik": 521, "kertas_kardus": 365,
          "saset_kemasan": 673, "styrofoam": 671}

root = pathlib.Path(DRIVE_ROOT)
if not root.exists():
    raise SystemExit(f"[GAGAL] {root} tidak ada - isi DRIVE_ROOT dengan folder dataset Anda.")

found = [d for d in root.rglob("train")
         if d.is_dir() and all((d / c).is_dir() for c in CLASSES)]
if not found:
    print("isi", root, ":", [str(p.relative_to(root)) for p in list(root.iterdir())[:20]])
    raise SystemExit("[GAGAL] tidak ketemu folder train/ berisi 5 kelas di bawah DRIVE_ROOT.")

src = max(found, key=lambda d: sum(len(list((d / c).glob("*.jpg"))) for c in CLASSES))
counts = {c: len(list((src / c).glob("*.jpg"))) for c in CLASSES}
print("dataset:", src, "| total", sum(counts.values()), "gambar")
beda = [c for c in CLASSES if counts[c] != EXPECT[c]]
for c in CLASSES:
    print(f"  {c:16s} {counts[c]:5d}  {'' if counts[c] == EXPECT[c] else '<-- beda dari snapshot split (%d)' % EXPECT[c]}")
if beda:
    print("\n[PERINGATAN] snapshot dataset ini beda dari yang dipakai membekukan splits/val.txt.")
    print("Kalau jumlahnya beda, make_split.py --check akan gagal - itu memang benar. Angka hasil")
    print("tidak akan sebanding dengan laptop lain. Pakai dulu snapshot yang sama (2753 gambar).")
else:
    print("\n[OK] snapshot dataset sama dengan yang membekukan splits/val.txt.")

link = pathlib.Path("train")
if link.is_symlink():
    link.unlink()
if not link.exists():
    link.symlink_to(src, target_is_directory=True)
print("train/ ->", link.resolve())

In [ ]:
!python make_split.py --check

> **Yang dilatih = kandidat, bukan submission.** `train.py --model {CANDIDATE}` membaca arsitektur **dari commit yang di-clone**, jadi kandidat itu wajib sudah ter-push lebih dulu. `submission/` sengaja tidak disentuh supaya pasangan `model.py`+`model.pth` di repo selalu sinkron dan bisa dievaluasi kapan saja.

`make_split.py --check` wajib `[OK]` (413 val / 2340 train, sha256 kanonik `b56ef897...`). Kalau `[GAGAL]`/`[BEDA]`, jangan lanjut melatih: berarti datasetnya beda dan hasilnya tidak bisa dibandingkan dengan laptop lain.


In [ ]:
!python train.py --model {CANDIDATE} --out {OUT_WEIGHTS} --epochs {EPOCHS} --bs {BATCH_SIZE} --workers {WORKERS}


`train.py` menulis bobot kandidat ke `runs/ens_20ep.pth` (+ `runs/last.json` berisi seluruh riwayat epoch), dan memilih checkpoint terbaik dengan **macro-F1 memakai metrik + urutan kelas dari `evaluate.py`**.

Selama ini `submission/` **tidak disentuh sama sekali** — promosi ke `submission/` baru dilakukan setelah kandidat terbukti menang.


In [ ]:
import pathlib, shutil

# evaluate.py memuat `submission/model.py` + `submission/model.pth` dengan strict=True, jadi
# kandidat diuji lewat folder sementara: model.py kandidat + bobot hasil latihan tadi.
cand = pathlib.Path("/tmp/candidate")
cand.mkdir(exist_ok=True)
shutil.copy(CANDIDATE, cand / "model.py")
shutil.copy(OUT_WEIGHTS, cand / "model.pth")
print("folder kandidat:", sorted(p.name for p in cand.iterdir()))

!python evaluate.py --submission /tmp/candidate --data ./my_val --out runs/eval_candidate.json


In [ ]:
import hashlib, pathlib

w = pathlib.Path(OUT_WEIGHTS)
print(f"{w}: {w.stat().st_size / 2**20:.1f} MB (batas teknis 500 MB, batas file GitHub 100 MB)")
print("sha256:", hashlib.sha256(w.read_bytes()).hexdigest())

from google.colab import files
files.download(OUT_WEIGHTS)


### Setelah selesai — promosi kandidat (hanya kalau menang)

Pembanding di split yang sama: **ResNet-18 15 epoch = 0,9373**. Kandidat dipakai untuk submission hanya kalau angkanya lebih tinggi.

Bobot yang diunduh ditaruh di `runs/` (di-.gitignore, jadi kandidat nggak pernah ikut ter-commit). Promosi = **satu commit atomik**, karena `evaluate.py` memuat `model.py` + `model.pth` dengan `strict=True` — kalau salah satu tertinggal, submission gagal dievaluasi (skor 0 di putaran itu):

```bash
# 1. uji kandidat di folder sementara (folder runs/ diabaikan git)
python -c "import pathlib,shutil; d=pathlib.Path('runs/candidate'); d.mkdir(exist_ok=True); shutil.copy('experiments/ensemble_model.py', d/'model.py'); shutil.copy('runs/ens_20ep.pth', d/'model.pth')"
python evaluate.py --submission runs/candidate --data ./my_val --out runs/eval_candidate.json

# 2. promosi: dua file, satu commit
cp experiments/ensemble_model.py submission/model.py
cp runs/ens_20ep.pth submission/model.pth
python evaluate.py --submission ./submission --data ./my_val     # wajib lolos SEBELUM commit
git add submission/model.py submission/model.pth
git commit -m "submission: ensemble ResNet-18 + EfficientNet-B0 (macro-F1 <skor>)"
git push
```
